In [21]:
import os
import json
import numpy as np
import pandas as pd
from astropy.time import Time
from IPython import get_ipython


def _load_search_runtime(search_dir='/home/msp25gd/ResearchProjectMSc/HR/search'):
    """Load ASSET helpers once in the current notebook kernel."""
    required = ['ASSET', 'to_reduce']
    missing = [name for name in required if name not in globals()]
    if len(missing) == 0:
        return

    ip = get_ipython()
    if ip is None:
        raise RuntimeError('This helper must run inside an IPython/Jupyter kernel.')

    original_cwd = os.getcwd()
    try:
        os.chdir(search_dir)
        for nb in ['asset.ipynb', 'reduce_name.ipynb']:
            nb_path = os.path.join(search_dir, nb)
            if not os.path.exists(nb_path):
                raise FileNotFoundError(f'Missing required notebook: {nb_path}')
            ip.run_line_magic('run', nb_path)
    finally:
        os.chdir(original_cwd)


def _row_to_datetime(row):
    """Convert available row time fields into a regular UTC datetime."""
    if 'MJD-OBS' in row.index and pd.notna(row['MJD-OBS']):
        try:
            mjd_val = float(row['MJD-OBS'])
            return pd.Timestamp(Time(mjd_val, format='mjd', scale='utc').to_datetime())
        except Exception:
            pass

    for col in ['Date', 'DATE-OBS', 'DATE_OBS', 'Datetime', 'DATE']:
        if col in row.index and pd.notna(row[col]):
            dt = pd.to_datetime(row[col], errors='coerce', utc=True)
            if pd.notna(dt):
                return pd.Timestamp(dt)

    return pd.NaT


def _row_duration_seconds(row):
    """Estimate observation duration in seconds from MJD-END/MJD-OBS or EXPTIME."""
    if 'MJD-OBS' in row.index and 'MJD-END' in row.index:
        if pd.notna(row['MJD-OBS']) and pd.notna(row['MJD-END']):
            try:
                duration_sec = (float(row['MJD-END']) - float(row['MJD-OBS'])) * 86400.0
                if np.isfinite(duration_sec) and duration_sec > 0:
                    return float(duration_sec)
            except Exception:
                pass

    if 'EXPTIME' in row.index and pd.notna(row['EXPTIME']):
        try:
            exp = float(row['EXPTIME'])
            if np.isfinite(exp) and exp > 0:
                return float(exp)
        except Exception:
            pass

    return np.nan


def get_star_detection_info(
    star_name,
    line='K',
    dataset_root='/home/msp25gd/ResearchProjectMSc/ResolutionHandling/processed_candidates/',
    param_path='/home/msp25gd/ResearchProjectMSc/HR/search/param.json',
    search_dir='/home/msp25gd/ResearchProjectMSc/HR/search',
):
    """
    Return three outputs for one star:
    1) summary dict with total spectra and number flagged with detection
    2) compact table for all spectra with object name and observation date
    3) compact table for detected spectra only with dip details
    """
    _load_search_runtime(search_dir=search_dir)

    if not os.path.exists(param_path):
        raise FileNotFoundError(f'param.json not found: {param_path}')
    if not os.path.isdir(dataset_root):
        raise FileNotFoundError(f'dataset_root not found: {dataset_root}')

    with open(param_path) as f:
        param = json.load(f)
    param['dataset'] = dataset_root if dataset_root.endswith('/') else dataset_root + '/'

    reduced = to_reduce(str(star_name)) if 'to_reduce' in globals() else str(star_name)
    star_path = os.path.join(param['dataset'], reduced)

    if not os.path.isdir(star_path):
        raise FileNotFoundError(
            f"Star folder not found in dataset: {star_path}. "
            f"Check reduced name '{reduced}' and dataset_root."
        )

    search = ASSET(parameters=param, line=line)
    search.ccf = False
    spec_param = search.spec_analysis(star_path + '/')

    if spec_param is None:
        raise ValueError(f'Not enough spectra to analyze detections for {reduced}.')

    new_spectra, med, med_err = spec_param

    if not hasattr(search, 'df') or search.df is None:
        raise RuntimeError('ASSET did not expose metadata table (search.df).')
    spectra_table = search.df.copy().reset_index(drop=True)

    if len(spectra_table) != len(new_spectra):
        raise RuntimeError(
            f'Row mismatch: metadata rows={len(spectra_table)} vs spectra={len(new_spectra)}'
        )

    detection_flags = []
    width_list = []
    min_sigma_list = []
    rv_at_min_list = []

    for idx, spec in enumerate(new_spectra):
        snr = search.snr(spec, med, search.spectra_err[idx], med_err)
        sd = np.std(snr)

        corr_snr = snr.copy()[search.snr_idxrange]
        sig = corr_snr / sd
        min_detect = float(np.nanmin(sig))

        filtered_rv = search.radial_velocity[search.snr_idxrange]
        rv_detect = float(filtered_rv[np.nanargmin(sig)])

        width = float(search.get_width(sig)) if min_detect < search.threshold else 0.0
        is_detection = (min_detect < search.threshold) and (width >= search.width_filter)

        detection_flags.append(bool(is_detection))
        width_list.append(width)
        min_sigma_list.append(min_detect)
        rv_at_min_list.append(rv_detect)

    object_col = 'Object' if 'Object' in spectra_table.columns else ('OBJECT' if 'OBJECT' in spectra_table.columns else 'Reduced')
    if object_col == 'Reduced' and 'Reduced' not in spectra_table.columns:
        spectra_table['Reduced'] = reduced

    spectra_table['ObjectName'] = spectra_table[object_col].astype(str)
    spectra_table['ObservationDateTimeUTC'] = spectra_table.apply(_row_to_datetime, axis=1)
    spectra_table['Detection'] = detection_flags
    spectra_table['DetectionWidth'] = width_list
    spectra_table['DetectionDurationSeconds'] = spectra_table.apply(_row_duration_seconds, axis=1)
    spectra_table['DipSignificanceSigma'] = min_sigma_list
    spectra_table['HeliocentricVelocity_km_s'] = rv_at_min_list

    all_observations_table = spectra_table[['ObjectName', 'ObservationDateTimeUTC']].copy()
    detected_only_table = spectra_table[spectra_table['Detection']][
        [
            'ObjectName',
            'ObservationDateTimeUTC',
            'HeliocentricVelocity_km_s',
            'DipSignificanceSigma',
            'DetectionWidth',
            'DetectionDurationSeconds',
        ]
    ].copy().reset_index(drop=True)

    summary = {
        'input_star': str(star_name),
        'reduced_star': str(reduced),
        'line': str(line),
        'total_spectra': int(len(all_observations_table)),
        'detected_spectra': int(len(detected_only_table)),
    }

    detection_times = detected_only_table['ObservationDateTimeUTC'].dropna().sort_values()
    if len(detection_times) > 0:
        detection_dates_text = ', '.join(dt.strftime('%Y-%m-%d %H:%M:%S UTC') for dt in detection_times)
    else:
        detection_dates_text = 'None'

    summary_text = (
        f"{reduced}: total spectra = {summary['total_spectra']}, "
        f"detected spectra = {summary['detected_spectra']}, "
        f"detection dates = {detection_dates_text}"
    )
    summary['summary_text'] = summary_text

    return summary, all_observations_table, detected_only_table


# Example usage:
# summary, all_spectra_table, detected_only_table = get_star_detection_info('hd22049')
# print(summary['summary_text'])
# display(all_spectra_table)
# display(detected_only_table)

In [23]:
summary, all_spectra_table, detected_only_table = get_star_detection_info('hd76131')
print(summary['summary_text'])
print(summary)
display(all_spectra_table)
display(detected_only_table)

hd76131: total spectra = 7, detected spectra = 1, detection dates = 2003-02-16 04:31:23 UTC
{'input_star': 'hd76131', 'reduced_star': 'hd76131', 'line': 'K', 'total_spectra': 7, 'detected_spectra': 1, 'summary_text': 'hd76131: total spectra = 7, detected spectra = 1, detection dates = 2003-02-16 04:31:23 UTC'}


,ObjectName,ObservationDateTimeUTC
0,HD-76131,2008-04-17 03:05:03.426432
1,HD_76131,2003-02-16 04:37:06.950784
2,HD_76131,2003-02-16 04:31:23.846016
3,HD_76131,2003-02-16 04:48:35.552736
4,HD_76131,2003-02-16 04:17:24.921888
5,HD_76131,2003-02-16 04:20:39.849792
6,HD_76131,2003-02-16 04:42:51.996096


,ObjectName,ObservationDateTimeUTC,HeliocentricVelocity_km_s,DipSignificanceSigma,DetectionWidth,DetectionDurationSeconds
0,HD_76131,2003-02-16 04:31:23.846016,152.224493,-3.667605,2.0,300.0011
